# 07 배경·제품·결함 국소 밝기 재검증

기존 01~06 결과를 보존한 확장이다. 실제 구현은 mv_analysis.background_brightness / regional_cli를 호출한다. NG PNG는 제품 마스크가 아니다. 자동 전경은 정답이 아니며 불확실하면 제품 밝기는 결측이다.

먼저 docs/BACKGROUND_BRIGHTNESS.md의 표본 검수를 수행한다. 전체 분석은 검수 근거와 체크포인트로 CLI에서 실행한다. 이 Notebook은 기본적으로 결과만 읽는다. 출력 제거 후 공유하고 개인 경로/서버 token을 저장하지 않는다.

In [ ]:
import os
from pathlib import Path
import pandas as pd
from IPython.display import display, Image
project = Path.cwd()
if not (project / "pyproject.toml").exists() and (project.parent / "pyproject.toml").exists():
    project = project.parent
run = Path(os.environ.get("MV_REGIONAL_RUN", str(project / "outputs/background_full_20261111_v1")))
required = run / "regional_validation.json"
if not required.exists():
    raise FileNotFoundError("영역별 집계가 없습니다. CLI 전체 계산/--finalize 상태와 MV_REGIONAL_RUN을 확인하세요.")
display(pd.read_json(required, typ="series"))

## 입력·출력과 검수 순서

입력: 같은 실행의 regional_validation.json, foreground_segmentation_metrics.csv, brightness_before_after_comparison.csv. 출력: 읽기용 표/그림이며 원본 JPG/PNG를 수정하지 않는다. 오류는 파일 누락, 미완료 집계, 환경변수 경로를 먼저 확인한다.

배경별 분포 → 분할 통과/불확실 비율 → paired 전체/제품 비교 → 국소 대비 → Train 전용 증강 후보 순서로 본다. method agreement는 GT IoU/Dice가 아니다.

In [ ]:
quality = pd.read_csv(run / "foreground_segmentation_metrics.csv")
display(quality.groupby(["category", "segmentation_status"]).size().unstack(fill_value=0))
comparison = pd.read_csv(run / "brightness_before_after_comparison.csv")
display(comparison.sort_values("absolute_difference_reduction", ascending=False).head(20))

whole_all은 전체 영상, whole_paired와 product는 동일한 품질 통과 영상이다. 제외에 따른 선택 효과를 분리해야 한다. NG/OK가 각2장 미만이면 차이와 d가 결측이다. 배경 제거 후 차이 감소는 분할/선택 오차의 영향도 받으므로 인과 효과로 단정하지 않는다.

In [ ]:
for name in ["regional_brightness_overview.png", "augmentation_review.png"]:
    if (run / name).exists():
        display(Image(filename=str(run / name)))
report = project / "reports/BACKGROUND_AWARE_BRIGHTNESS_ANALYSIS.md"
print("해석과 재현 명령:", report.name)